# Task 3: Chatbot using Hugging Face Transformers


In [ ]:
#installing required libraries (already exists in my environment)
!pip install transformers torch --quiet

#### Importing libraries

In [1]:
#import libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

c:\Users\Dell\miniforge3\envs\mix\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model Loading

In [ ]:
model_name = "microsoft/DialoGPT-medium"  

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, use_safetensors=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 293/293 [00:00<00:00, 1140.48it/s]


### Chatbot Build


In [ ]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("(Type 'exit' or 'quit' to end)\n")

chat_history_ids = None

#for continous conversion
while True: 
    user_input = input("You: ").strip()

    if not user_input:
        print("Chatbot: Please type something! \n")
        continue

    #exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day! ")
        break

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    #append new input to conversation history
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    if bot_input_ids.shape[-1] > 1000:
        bot_input_ids = bot_input_ids[:, -1000:]

    #generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()


    if not response:
        response = "I'm not sure about that. Could you rephrase?"

    print(f"Chatbot: {response}\n")

Chatbot: Hello! I am your AI assistant. How can I help you today?
(Type 'exit' or 'quit' to end)

Chatbot: Hi, it's nice to meet you. I'm Aesop

Chatbot: I love lamp!

Chatbot: ohhh my bad i meant ai lol sorry for the confusion lt 3

Chatbot: A machine with intelligence?

Chatbot: The people that were around before we got there

Chatbot: that was a great question actually xD so many good questions today in this sub :P

Chatbot: Goodbye! Have a great day! 


### Conclusion

The model used in this is  Hugging Face's DialoGPT-medium model,which generates responses that can hold conversations with user . It helped me to understand how transformer models work in real-world conversational AI applications.